# 25. CNN에서 Transformer로

이 노트북은 `24_사전학습_Segmentation_모델_실습.ipynb` 다음 단계로, CNN 중심의 비전 모델에서 Transformer 기반 비전 모델로 넘어갈 때 무엇이 달라지는지 정리합니다.

CNN은 이미지의 지역 패턴을 효율적으로 찾는 구조적 가정을 가지고 있습니다. 반면 Transformer는 입력을 token sequence로 보고, token 사이의 관계를 attention으로 직접 계산합니다.

이번 노트북의 목표는 다음과 같습니다.

- CNN의 inductive bias가 무엇인지 이해합니다.
- Transformer가 이미지를 다루려면 이미지가 sequence로 바뀌어야 함을 이해합니다.
- convolution과 attention이 정보를 섞는 방식의 차이를 비교합니다.
- ViT로 넘어가기 전에 patch, token, global relation의 관점을 준비합니다.

## 25-1. 준비

외부 데이터 없이 실행할 수 있도록 작은 격자 이미지를 직접 만듭니다. 이 노트북의 코드는 실제 모델 학습보다 개념 시각화에 초점을 둡니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 25-2. CNN의 기본 가정

CNN은 이미지에 대해 다음과 같은 강한 가정을 사용합니다.

- 가까운 픽셀끼리는 서로 관련이 깊습니다.
- 같은 filter를 여러 위치에 공유해서 적용할 수 있습니다.
- 작은 local pattern을 쌓아 더 큰 pattern을 만들 수 있습니다.

이런 가정을 **inductive bias**라고 부릅니다. 데이터가 적거나 이미지의 지역 구조가 중요할 때 CNN이 강력한 이유입니다.

In [ ]:
image = np.zeros((8, 8))
image[1:3, 1:3] = 0.4
image[2:6, 4:6] = 0.8
image[6, 1:7] = 0.6

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image, cmap='Blues', vmin=0, vmax=1)
ax.set_title('작은 local pattern을 보는 CNN')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.grid(color='white', linewidth=1.5)

kernel = Rectangle((3.5, 1.5), 3, 3, fill=False, edgecolor='crimson', linewidth=3)
ax.add_patch(kernel)
ax.text(5, 1.1, '3 x 3 filter', color='crimson', ha='center', weight='bold')
plt.show()

CNN의 filter는 한 번에 작은 영역만 봅니다. 깊은 층을 쌓으면 receptive field가 커지지만, 기본 연산 자체는 지역적입니다.

반대로 Transformer의 attention은 모든 token 쌍의 관계를 계산할 수 있습니다. 따라서 멀리 떨어진 위치의 관계도 한 layer 안에서 직접 연결할 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax in axes:
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 3)
    ax.axis('off')

axes[0].set_title('Convolution: 주로 이웃을 섞음')
for x in range(5):
    axes[0].scatter(x + 0.5, 1.5, s=400, color='#dbeafe', edgecolor='#2563eb')
    axes[0].text(x + 0.5, 1.5, f'p{x}', ha='center', va='center')
for x in range(4):
    axes[0].add_patch(FancyArrowPatch((x + 0.65, 1.5), (x + 1.35, 1.5), arrowstyle='<->', mutation_scale=12, color='#2563eb'))

axes[1].set_title('Attention: 모든 token 관계를 볼 수 있음')
points = [(0.8, 1.5), (2.5, 2.4), (4.2, 1.5), (2.5, 0.6)]
for i, (x, y) in enumerate(points):
    axes[1].scatter(x, y, s=450, color='#dcfce7', edgecolor='#16a34a')
    axes[1].text(x, y, f't{i}', ha='center', va='center')
for i, (x1, y1) in enumerate(points):
    for j, (x2, y2) in enumerate(points):
        if i < j:
            axes[1].plot([x1, x2], [y1, y2], color='#16a34a', alpha=0.35)

plt.show()

## 25-3. 이미지를 sequence로 본다는 뜻

Transformer는 원래 문장처럼 token의 나열을 처리하는 구조입니다. 이미지를 Transformer에 넣으려면 이미지를 작은 patch로 나누고, 각 patch를 하나의 token처럼 다루어야 합니다.

```text
image -> patches -> patch embeddings -> token sequence -> Transformer encoder
```

이 관점 전환이 Vision Transformer의 출발점입니다.

In [ ]:
grid_size = 4
patch_values = np.arange(grid_size * grid_size).reshape(grid_size, grid_size)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(patch_values, cmap='viridis')
axes[0].set_title('이미지 patch')
axes[0].set_xticks(range(grid_size))
axes[0].set_yticks(range(grid_size))
axes[0].grid(color='white', linewidth=2)
for y in range(grid_size):
    for x in range(grid_size):
        axes[0].text(x, y, f'P{patch_values[y, x]}', color='white', ha='center', va='center', weight='bold')

axes[1].set_title('patch sequence')
axes[1].set_xlim(-0.5, 15.5)
axes[1].set_ylim(-0.5, 1.5)
axes[1].axis('off')
for i in range(16):
    rect = Rectangle((i - 0.35, 0.35), 0.7, 0.5, facecolor='#fef3c7', edgecolor='#d97706')
    axes[1].add_patch(rect)
    axes[1].text(i, 0.6, f'P{i}', ha='center', va='center', fontsize=9)

plt.show()

## 25-4. CNN과 Transformer의 비교

| 관점 | CNN | Transformer |
|---|---|---|
| 입력 해석 | 2D grid | token sequence |
| 기본 연산 | convolution | self-attention |
| 정보 혼합 | 가까운 영역부터 점진적으로 | 모든 token 관계를 직접 계산 |
| 강한 가정 | local pattern, translation equivariance | 상대적으로 약한 구조적 가정 |
| 장점 | 데이터 효율, 계산 효율 | 전역 관계 모델링, 확장성 |

Transformer가 항상 CNN보다 좋은 것은 아닙니다. Transformer는 강한 구조적 가정이 적은 대신, 충분한 데이터와 계산량에서 좋은 확장성을 보이는 경우가 많습니다.

## 정리

- CNN은 이미지의 지역성과 위치 공유라는 inductive bias를 적극적으로 사용합니다.
- Transformer는 입력을 token sequence로 보고 token 사이 관계를 attention으로 계산합니다.
- 이미지를 Transformer에 넣으려면 patch token으로 바꾸는 관점 전환이 필요합니다.
- 다음 노트북 `26_Attention_기초_복습.ipynb`에서는 query, key, value와 self-attention 계산을 복습합니다.